In [0]:
df_airports = spark.read.csv("/Volumes/practicalbigdata/dataanalyst/bigdata-files/airports.csv",header=True,inferSchema=True)

In [0]:
df_runways = spark.read.csv("/Volumes/practicalbigdata/dataanalyst/bigdata-files/runways.csv",header=True,inferSchema=True)

In [0]:
from pyspark.sql.functions import col, count, avg, sum, max

In [0]:
df_airports = df_airports.withColumn("latitude_deg", col("latitude_deg").cast("double")) \
                         .withColumn("longitude_deg", col("longitude_deg").cast("double")) \
                         .withColumn("elevation_ft", col("elevation_ft").cast("int"))

df_runways = df_runways.withColumn("length_ft", col("length_ft").cast("int")) \
                       .withColumn("width_ft", col("width_ft").cast("int"))

In [0]:
df_airports.show(10)
df_runways.show(10)

+------+-----+-------------+--------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------------+--------+
|    id|ident|         type|                name|      latitude_deg|      longitude_deg|elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|           home_link|      wikipedia_link|keywords|
+------+-----+-------------+--------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------------+--------+
|  6523|  00A|     heliport|   Total RF Heliport|         40.070985|         -74.933689|          11|       NA|         US|     US-PA|    Bensalem|               no|     NULL|     NULL|    K00A|       00A|https://www.pennd...|   

In [0]:
df_airports.printSchema()
print("Airports row count:", df_airports.count())

root
 |-- id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

Airports row count: 85324


In [0]:
df_runways.printSchema()
print("Runways row count:", df_runways.count())

root
 |-- id: integer (nullable = true)
 |-- airport_ref: integer (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = true)
 |-- surface: string (nullable = true)
 |-- lighted: integer (nullable = true)
 |-- closed: integer (nullable = true)
 |-- le_ident: string (nullable = true)
 |-- le_latitude_deg: double (nullable = true)
 |-- le_longitude_deg: double (nullable = true)
 |-- le_elevation_ft: integer (nullable = true)
 |-- le_heading_degT: double (nullable = true)
 |-- le_displaced_threshold_ft: integer (nullable = true)
 |-- he_ident: string (nullable = true)
 |-- he_latitude_deg: double (nullable = true)
 |-- he_longitude_deg: double (nullable = true)
 |-- he_elevation_ft: integer (nullable = true)
 |-- he_heading_degT: double (nullable = true)
 |-- he_displaced_threshold_ft: integer (nullable = true)

Runways row count: 47884


In [0]:
df_combined = df_airports.join(
    df_runways,
    df_airports["ident"] == df_runways["airport_ident"],
    "inner"
).select(
    df_airports["*"],
    df_runways["airport_ref"],
    df_runways["airport_ident"],
    df_runways["length_ft"],
    df_runways["width_ft"],
    df_runways["surface"],
    df_runways["lighted"]
)

df_combined.printSchema()
print("Combined row count:", df_combined.count())

root
 |-- id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- airport_ref: integer (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = true)
 |-- surface: string (nullable = tr

In [0]:
df_combined = df_airports.join(
    df_runways,
    df_airports["ident"] == df_runways["airport_ident"],
    "inner"
).select(
    df_airports["*"],
    df_runways["airport_ref"],
    df_runways["airport_ident"],
    df_runways["length_ft"],
    df_runways["width_ft"],
    df_runways["surface"],
    df_runways["lighted"]
)df_combined.groupBy("iso_country") \
    .agg(count("*").alias("runway_count")) \
    .orderBy(col("runway_count").desc()) \
    .show(10)

+-----------+------------+
|iso_country|runway_count|
+-----------+------------+
|         US|       26450|
|         BR|        5825|
|         AU|        2103|
|         CA|        1578|
|         AR|         761|
|         FR|         659|
|         GB|         600|
|         DE|         592|
|         RU|         489|
|         ID|         414|
+-----------+------------+
only showing top 10 rows


In [0]:
df_combined.filter(col("length_ft").isNotNull()) \
    .groupBy("continent") \
    .agg(avg("length_ft").alias("avg_runway_length")) \
    .orderBy(col("avg_runway_length").desc()) \
    .show()

+---------+------------------+
|continent| avg_runway_length|
+---------+------------------+
|       AN| 7768.857142857143|
|       AS|6925.6469996647675|
|       AF| 6787.526448362721|
|       EU| 4103.151335311572|
|       OC|3558.0401785714284|
|       SA|2857.0891141239963|
|       NA| 2599.724218941758|
+---------+------------------+



In [0]:
df_combined.filter((col("iso_country") == "IN") & (col("lighted") == 1)) \
    .groupBy(df_airports["name"], "municipality") \
    .agg(count("*").alias("lighted_runways")) \
    .orderBy(col("lighted_runways").desc()) \
    .show()

+--------------------+------------+---------------+
|                name|municipality|lighted_runways|
+--------------------+------------+---------------+
|Indira Gandhi Int...|   New Delhi|              3|
|Navi Mumbai Inter...| Navi Mumbai|              2|
|Raja Bhoj Interna...|      Bhopal|              2|
|Rajiv Gandhi Inte...|   Hyderabad|              2|
|Netaji Subhash Ch...|     Kolkata|              2|
|Agra Airport / Ag...|        Agra|              2|
|   Prayagraj Airport|   Allahabad|              2|
|Ambala Air Force ...|      Ambala|              2|
|Chennai Internati...|     Chennai|              2|
|Kempegowda Intern...|   Bengaluru|              2|
|Mangaluru Interna...|   Mangaluru|              2|
|Bidar Airport / B...|       Bidar|              2|
|Chhatrapati Shiva...|      Mumbai|              2|
|Tambaram Air Forc...|     Chennai|              2|
|       Jammu Airport|       Jammu|              1|
|      Dhulia Airport|        NULL|              1|
|   Shravast

In [0]:
df_combined.groupBy(df_airports["name"], "iso_country") \
    .agg(max("length_ft").alias("max_runway_length")) \
    .orderBy(col("max_runway_length").desc()) \
    .show(5)

+--------------------+-----------+-----------------+
|                name|iso_country|max_runway_length|
+--------------------+-----------+-----------------+
|Gunflint Seaplane...|         US|            30000|
|Libby Camps Seapl...|         US|            26000|
|Long Lake Seaplan...|         US|            25000|
|Brookville Reserv...|         US|            25000|
|Conchas Lake Seap...|         US|            21120|
+--------------------+-----------+-----------------+
only showing top 5 rows


In [0]:
df_combined.write.mode("overwrite").parquet("/Volumes/practicalbigdata/dataanalyst/bigdata-files/df_combined_parquet")

In [0]:
df_saved = spark.read.parquet("/Volumes/practicalbigdata/dataanalyst/bigdata-files/df_combined_parquet")

df_saved.show(10)

+------+-----+-------------+--------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------------+--------+-----------+-------------+---------+--------+-------+-------+
|    id|ident|         type|                name|      latitude_deg|      longitude_deg|elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|           home_link|      wikipedia_link|keywords|airport_ref|airport_ident|length_ft|width_ft|surface|lighted|
+------+-----+-------------+--------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------------+--------+-----------+-------------+---------+--------+-------+-------+
|  6523|  00A|     heliport|   Total RF Helipo